# Heat pump performance

In this exercise you will estimate the annual heat load of a building as well as determine how much of the load can be covered by an air-to-air heat pump. The case will be based on Tina's house - so it's a real system!

The first step is to import a few Python pacakges.

In [ ]:
# Install pvlib in Google Colab as this is not a standard package.
!pip install pvlib

In [8]:
import pvlib  # library for retrieving weather data & modeling photovolatics
import pandas as pd  # library for data analysis
import matplotlib.pyplot as plt  # library for plotting
import numpy as np  # library for math and linear algebra

## Step 0: Investigate heat pump datasheet

You can find the datasheet for Tina's heat pump [here](https://heatnow.dk/wp-content/uploads/2021/09/Specification_Sheet_9404.pdf).


Determine the following characteristics:
- What is the maxium and minimum heating capacity?
- What is the maximum and minimum COP?

## Step 1: Define a location

A location is defined by a latitude and longitude according to the convention of [ISO 6709](https://en.wikipedia.org/wiki/ISO_6709). Specifically, latitude is in degrees north of the equator and the longitude is in degrees east of the prime meridian.

The coordinates corresponds to Sisimiut.

In [9]:
latitude = 66.9343
longitude = -53.6748

## Step 2: Retrieve irradiance data from NASA POWER

In this step, weather data is retrieved from the NASA POWER [dataset](https://power.larc.nasa.gov/data-access-viewer/) using the pvlib function [``get_nasa_power``](https://pvlib-python.readthedocs.io/en/latest/reference/generated/pvlib.iotools.get_nasa_power.html).

👉 Simply execute this cell without making any changes.

In [10]:
# Define start and end date
start = "2025-01-01"
end = "2025-12-31"

parameters = parameters=['temp_air']

data, meta = pvlib.iotools.get_nasa_power(
    latitude, longitude, start, end, parameters)

data

,temp_air
2025-01-01 00:00:00+00:00,-13.29
2025-01-01 01:00:00+00:00,-13.19
2025-01-01 02:00:00+00:00,-12.90
2025-01-01 03:00:00+00:00,-12.58
2025-01-01 04:00:00+00:00,-12.22
...,...
2025-12-31 19:00:00+00:00,0.52
2025-12-31 20:00:00+00:00,0.03
2025-12-31 21:00:00+00:00,-0.14
2025-12-31 22:00:00+00:00,-0.06


## Step 3: Plot the temperature

👉 In the code cell below, plot the ambient temperature for the full year.

In [1]:
# Write your code here


## Step 4: Estimate heat load

The ambient temperature is the main driver of heat losses from a house.

We will be simulating Tina's house, which is a single family house which approximately has a heat loss coefficient of 140 W/K. Remember how to estimate this from Martin's lecture?

👉 Calculate the heat load for the year assuming an indoor temperature of 20 °C.

*Hint: remember that you cannot have negative heat load. You can use ``.clip(lower=0)`` to remove negative values.*





In [ ]:
# Write your code here
heatload = 

## Step 5: Heat pump capacity

In this step we'll calculate the maximum heat pump capacity for each time step. The capacity depends on the source temperature (ambient air) and the supply temperature (indoor supply air temperature). We will assume a supply temperature of 20 degrees C (which corresponds to the values given in the datasheet).


👉 Skip the below complicated code cell and move on to the next step.

In [ ]:
from __future__ import annotations
import warnings
import pandas as pd

# ---------------------------------------------------------------------------
# Anchor points -- (T_outdoor_db C, capacity_kW, COP, source)
# ---------------------------------------------------------------------------
_ANCHORS: list[tuple[float, float, float, str]] = [
    (-25.0, 3.600, 2.220, "datasheet-powerful"),
    (-20.0, 4.200, 2.400, "datasheet-powerful"),
    (-15.0, 4.780, 2.540, "datasheet-powerful"),
    ( -7.0, 5.000, 2.580, "datasheet-powerful"),
    (  7.0, 7.500, 3.969, "carnot-extrap-capped"),
    ( 10.0, 7.500, 4.487, "carnot-extrap-capped"),
    ( 15.0, 7.500, 5.733, "carnot-extrap-capped"),
]

_POINTS: list[tuple[float, float, float]] = [
    (t, q, c) for t, q, c, _ in _ANCHORS
]

CAP_MAX_KW: float = 7.50
T_MIN: float = _POINTS[0][0]   # -25 C
T_MAX: float = _POINTS[-1][0]  # +15 C
T_INDOOR_REF: float = 20.0     # C DB -- datasheet indoor reference


def _interp(x: float, x0: float, x1: float, y0: float, y1: float) -> float:
    """Linear interpolation."""
    return y0 + (y1 - y0) * (x - x0) / (x1 - x0)


def _lookup_scalar(t: float) -> tuple[float, float]:
    """Return (capacity_kW, COP) for a single clamped outdoor temperature."""
    t = max(T_MIN, min(T_MAX, t))
    for i in range(len(_POINTS) - 1):
        t0, q0, c0 = _POINTS[i]
        t1, q1, c1 = _POINTS[i + 1]
        if t0 <= t <= t1:
            return _interp(t, t0, t1, q0, q1), _interp(t, t0, t1, c0, c1)
    return _POINTS[-1][1], _POINTS[-1][2]


def _carnot_cop(t_cond_air: float, t_evap_air: float) -> float:
    """Ideal Carnot COP from air-side temperatures."""
    t_cond = t_cond_air + 5.0 + 273.15   # K
    t_evap = t_evap_air - 8.0 + 273.15   # K
    return t_cond / max(t_cond - t_evap, 1.0)


def heating_performance(
    t_outdoor: pd.Series,
    t_indoor: float = T_INDOOR_REF,
) -> pd.DataFrame:
    """
    Estimate full-load heating performance for the Panasonic CU-HZ25XKE.

    Parameters
    ----------
    t_outdoor : pd.Series
        Outdoor air dry-bulb temperature in C, with any pandas-compatible
        index (typically a DatetimeIndex from a TMY or measured dataset).
        Valid range: -25 to +15 C. Values outside this range are clamped
        and a warning is issued.
    t_indoor : float, optional
        Indoor air dry-bulb temperature in C.
        Default is 20 C (the EN14825 datasheet reference condition).
        Deviations are corrected via Carnot lift scaling. Use with caution
        beyond +/-5 C from reference.

    Returns
    -------
    pd.DataFrame
        Index  : inherited from t_outdoor (e.g. DatetimeIndex)
        Columns: capacity_kw, cop, input_power_w

    Examples
    --------
    >>> import pandas as pd
    >>> tmy = pd.read_csv("tmy.csv", index_col=0, parse_dates=True)
    >>> df = heating_performance(tmy["T2m"])
    >>> df.resample("ME").mean()          # monthly averages
    >>> df["input_power_w"].sum() / 1e6  # total MWh consumed
    """
    if not isinstance(t_outdoor, pd.Series):
        raise TypeError(
            f"t_outdoor must be a pd.Series, got {type(t_outdoor).__name__}. "
            "Wrap plain arrays with pd.Series() before calling."
        )

    temps = t_outdoor.to_numpy(dtype=float)

    below_mask = temps < T_MIN
    above_mask = temps > T_MAX
    if below_mask.any():
        warnings.warn(
            f"{below_mask.sum()} value(s) below minimum ({T_MIN} C). "
            f"Clamped to {T_MIN} C -- results may be optimistic.",
            UserWarning, stacklevel=2,
        )
    if above_mask.any():
        warnings.warn(
            f"{above_mask.sum()} value(s) above maximum ({T_MAX} C). "
            f"Clamped to {T_MAX} C.",
            UserWarning, stacklevel=2,
        )

    use_indoor_correction = t_indoor != T_INDOOR_REF

    capacity_kw   = []
    cop_vals      = []
    input_power_w = []

    for t in temps:
        cap, cop = _lookup_scalar(t)

        if use_indoor_correction:
            scale = _carnot_cop(t_indoor, t) / _carnot_cop(T_INDOOR_REF, t)
            cap = min(cap * scale, CAP_MAX_KW)
            cop = cop * scale

        capacity_kw.append(round(cap, 3))
        cop_vals.append(round(cop, 3))
        input_power_w.append(round((cap / cop) * 1000.0, 1))

    return pd.DataFrame(
        {
            "capacity_kw":   capacity_kw,
            "cop":           cop_vals,
            "input_power_w": input_power_w,
        },
        index=t_outdoor.index,
    )



In [22]:
performance = heating_performance(data["temp_air"])

performance

C:\Users\arajen\AppData\Local\Temp\ipykernel_35772\2914464741.py:1: UserWarning: 10 value(s) below minimum (-25.0 C). Clamped to -25.0 C -- results may be optimistic.
  performance = heating_performance(data["temp_air"])


,capacity_kw,cop,input_power_w
2025-01-01 00:00:00+00:00,4.827,2.549,1894.0
2025-01-01 01:00:00+00:00,4.830,2.549,1894.7
2025-01-01 02:00:00+00:00,4.838,2.550,1896.8
2025-01-01 03:00:00+00:00,4.847,2.552,1899.0
2025-01-01 04:00:00+00:00,4.856,2.554,1901.6
...,...,...,...
2025-12-31 19:00:00+00:00,6.343,3.326,1907.0
2025-12-31 20:00:00+00:00,6.255,3.277,1908.6
2025-12-31 21:00:00+00:00,6.225,3.261,1909.2
2025-12-31 22:00:00+00:00,6.239,3.269,1908.9


## Step 6: Plot the heat capacity, cop, and electricity consumption?

The data is in the ``performance`` dataframe.

In [18]:
# Write your code here


## Step 7: Calculate heat pump heat supply

Now we know the heat load and the maximum heat capacity of the heat pump.

Whenever the heat load is greater than the heat capacity we need to use an alternative heating source (direct electric or oil boiler).

👉Determine what percentage of the heating load can be met by the heat pump?

In [ ]:
# Write your code


## Wrap up

Think about how this study was simplified? What assumptions were made?
